# NB12 Reproducibility Runs — 30 Independent Hospital-Out Partitions

**Project:** Evolving Interpretable Sepsis Mortality Risk Scores via Genetic Programming
**Notebook:** NB12_repro_runs — Reproducibility analysis (supplementary to NB12)

> **Updated for BCE loss (v2_bce); N_RUNS 10 → 30.** Old outputs reflect c=23/seed 5. New canonical: c=24/seed 14. All paths updated to `results/v2_bce/...`. GP predictions now use sigmoid (not clip).

## Purpose

Checks whether NB12's hospital-out result (RQ2) is specific to the one canonical 52/13 hospital partition, or whether it replicates across different choices of which 13 hospitals are held out. Each of the 30 runs draws a fresh random 52/13 hospital partition, retrains LR/RF/XGB/LR_platt/RF_platt on the new training set, and applies the **canonical GP formula (c=24, seed=14) zero-shot** — no retraining, no recalibration.

## Why GP is zero-shot here, unlike NB11_repro_runs

NB12's claim is zero-shot transfer of one fixed formula. Retraining GP per partition would test a different question. Holding GP fixed and varying the partition tests whether the *calibration stability claim* holds regardless of which hospitals are held out — which is the actual reproducibility question for RQ2. No PySR calls needed → this notebook runs in minutes.

## Inputs

| File | Description |
|---|---|
| `data/processed/features_curated.parquet` | Full feature matrix — stable across all 30 runs |
| `data/processed/feature_config.json` | MODELLING_COLS (58) and GP_TERMINALS (26) |
| `data/processed/apache4_predictions.csv` | APACHE-IV predicted mortality |
| `results/v2_bce/gp_runs/run_14_model.pkl` | Canonical GP model (seed=14, c=24) — held fixed, applied zero-shot every run |

## Outputs

| File | Description |
|---|---|
| `results/v2_bce/reproducibility_runs_hospout/run_01/` ... `run_30/` | Per-run artefacts |
| `results/v2_bce/tables/NB12_30run_reproducibility.csv` | Master aggregated table — 30 runs x 7 models |

## Run-to-seed mapping

`run_NN` (NN = 01..30) uses `random_state = NN - 1` (seeds 0–29).

In [1]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import json, pickle, time, sys, warnings
warnings.filterwarnings("ignore")

_nb_dir   = Path().resolve()
PROJECT   = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA_PROC = PROJECT / "data" / "processed"
RESULTS_VERSION = "v2_bce"
TABLES    = PROJECT / "results" / RESULTS_VERSION / "tables"
RUNS_DIR  = PROJECT / "results" / RESULTS_VERSION / "gp_runs"
REPRO_DIR = PROJECT / "results" / RESULTS_VERSION / "reproducibility_runs_hospout"
REPRO_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece, compute_metrics, calibration_bins

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
with open(DATA_PROC / "feature_config.json") as f:
    cfg = json.load(f)
MODELLING_COLS = cfg["MODELLING_COLS"]
GP_TERMINALS   = cfg["GP_TERMINALS"]
apache_preds   = pd.read_csv(DATA_PROC / "apache4_predictions.csv")
apache_map     = (apache_preds.dropna(subset=["apache4_pred"])
                   .set_index("patientunitstayid")["apache4_pred"])

ALL_HOSPITALS = np.sort(feat["hospitalid"].unique())
N_TEST_HOSP   = 13

# ── Canonical GP model (seed=14, c=24) — loaded once, applied zero-shot every run
with open(RUNS_DIR / "run_14_model.pkl", "rb") as fh:
    gp_model = pickle.load(fh)
GP_COMPLEXITY = 24
eq_idx = gp_model.equations_[
    gp_model.equations_["complexity"] == GP_COMPLEXITY
].index[0]
canonical_eq = str(gp_model.equations_.loc[eq_idx, "equation"])

print(f"Results version  : {RESULTS_VERSION}")
print(f"Feature frame    : {feat.shape}")
print(f"Hospitals (total): {len(ALL_HOSPITALS)}")
print(f"REPRO_DIR        : {REPRO_DIR}")
print()
print(f"Canonical GP expression (seed=14, c={GP_COMPLEXITY}, held fixed across all 30 runs):")
print(f"  {canonical_eq}")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Results version  : v2_bce
Feature frame    : (11164, 61)
Hospitals (total): 65
REPRO_DIR        : C:\ML PROJECT\sepsis-gp\results\v2_bce\reproducibility_runs_hospout

Canonical GP expression (seed=14, c=24, held fixed across all 30 runs):
  (bun / pf_ratio) - (((sqrt(lactate_max) + max(vent, intubated)) * -0.8085511) + ((min(platelets_min, map_mean) / age_numeric) - ((temperature - bilirubin) * -0.06636052)))


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# 30-run hospital-out reproducibility loop (resume-safe).
# No PySR calls — GP held fixed (canonical c=24, seed=14).
# ════════════════════════════════════════════════════════════════════════════
N_RUNS = 30

print("30-Run Hospital-Out Reproducibility (BCE loss, GP zero-shot)")
print("  Each run: fresh random 52/13 hospital partition -> retrain LR/RF/XGB/LR_platt/RF_platt")
print("            -> apply canonical GP formula zero-shot (sigmoid, not clip) -> evaluate")
print("  Resume-safe: completed runs are skipped automatically")
print()

def _logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1.0 - p))

marathon_t0 = time.time()
master_rows = []

for run_idx in range(N_RUNS):
    run_seed = run_idx
    run_name = f"run_{run_idx + 1:02d}"
    run_dir  = REPRO_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / "run_metrics.csv"

    if metrics_path.exists():
        print(f"{run_name} (seed={run_seed}): already complete — skipping")
        existing = pd.read_csv(metrics_path)
        existing["run"]  = run_idx + 1
        existing["seed"] = run_seed
        master_rows.append(existing)
        continue

    run_t0 = time.time()
    print(f"{run_name} (seed={run_seed}): starting ...", flush=True)

    rng = np.random.RandomState(run_seed)
    shuffled = rng.permutation(ALL_HOSPITALS)
    test_hospitals  = set(shuffled[:N_TEST_HOSP])
    train_hospitals = set(shuffled[N_TEST_HOSP:])

    split_out = feat[["patientunitstayid", "hospitalid"]].copy()
    split_out["split"] = np.where(split_out["hospitalid"].isin(test_hospitals), "test", "train")
    split_out.to_csv(run_dir / "hospital_split.csv", index=False)

    train_df = feat[feat["hospitalid"].isin(train_hospitals)].reset_index(drop=True)
    test_df  = feat[feat["hospitalid"].isin(test_hospitals)].reset_index(drop=True)

    X_train = train_df[MODELLING_COLS].to_numpy(dtype=np.float64)
    y_train = train_df["hospital_mortality"].to_numpy(dtype=np.float64)
    X_test  = test_df[MODELLING_COLS].to_numpy(dtype=np.float64)
    y_test  = test_df["hospital_mortality"].to_numpy(dtype=np.float64)
    X_test_gp = test_df[GP_TERMINALS].copy()

    print(f"  Train: {len(train_df):,} pts / {len(train_hospitals)} hosp "
          f"({y_train.mean()*100:.2f}%)   Test: {len(test_df):,} pts / {len(test_hospitals)} hosp "
          f"({y_test.mean()*100:.2f}%)")

    spw = (y_train == 0).sum() / (y_train == 1).sum()

    def make_lr(seed):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed)),
        ])

    def make_rf(seed):
        return RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5,
            class_weight="balanced", random_state=seed, n_jobs=-1)

    lr_model = make_lr(run_seed);  lr_model.fit(X_train, y_train)
    rf_model = make_rf(run_seed);  rf_model.fit(X_train, y_train)
    xgb_model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                               scale_pos_weight=spw, random_state=run_seed,
                               eval_metric="logloss", verbosity=0)
    xgb_model.fit(X_train, y_train)

    lr_pred  = lr_model.predict_proba(X_test)[:, 1]
    rf_pred  = rf_model.predict_proba(X_test)[:, 1]
    xgb_pred = xgb_model.predict_proba(X_test)[:, 1]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=run_seed)
    lr_oof = cross_val_predict(make_lr(run_seed), X_train, y_train, cv=skf,
                                method="predict_proba", n_jobs=-1)[:, 1]
    rf_oof = cross_val_predict(make_rf(run_seed), X_train, y_train, cv=skf,
                                method="predict_proba", n_jobs=-1)[:, 1]

    lr_platt = LogisticRegression(max_iter=1000)
    lr_platt.fit(_logit(lr_oof).reshape(-1, 1), y_train)
    rf_platt = LogisticRegression(max_iter=1000)
    rf_platt.fit(_logit(rf_oof).reshape(-1, 1), y_train)

    lr_platt_pred = lr_platt.predict_proba(_logit(lr_pred).reshape(-1, 1))[:, 1]
    rf_platt_pred = rf_platt.predict_proba(_logit(rf_pred).reshape(-1, 1))[:, 1]

    # ── GP zero-shot: sigmoid (not clip) — matches NB10/NB12 Cell 3 design ───────
    gp_raw  = gp_model.predict(X_test_gp, index=eq_idx)
    gp_raw  = np.where(np.isfinite(gp_raw), gp_raw, 0.0)
    gp_pred = sigmoid(gp_raw)

    test_preds = {
        "GP": gp_pred, "LR": lr_pred, "RF": rf_pred, "XGB": xgb_pred,
        "LR_platt": lr_platt_pred, "RF_platt": rf_platt_pred,
    }

    ap_pred = np.array([apache_map.get(pid, np.nan) for pid in test_df["patientunitstayid"]])
    ap_mask = ~np.isnan(ap_pred)

    run_metric_rows = []
    for name in ["GP", "LR", "RF", "XGB", "LR_platt", "RF_platt"]:
        m = compute_metrics(y_test, test_preds[name], label=name)
        m["model"] = name
        run_metric_rows.append(m)
    m_ap = compute_metrics(y_test[ap_mask], np.clip(ap_pred[ap_mask], 1e-7, 1-1e-7), label="APACHE-IV")
    m_ap["model"] = "APACHE-IV"
    run_metric_rows.append(m_ap)
    run_metrics_df = pd.DataFrame(run_metric_rows)
    run_metrics_df.to_csv(metrics_path, index=False)

    preds_out = test_df[["patientunitstayid", "hospitalid", "hospital_mortality"]].copy()
    for name in ["GP", "LR", "RF", "XGB", "LR_platt", "RF_platt"]:
        preds_out[name.lower() + "_pred"] = test_preds[name]
    preds_out.to_csv(run_dir / "model_predictions.csv", index=False)

    elapsed = time.time() - run_t0
    print(f"  Test metrics (AUROC/ECE): " +
          "  ".join(f"{r['model']}={r['auroc']:.3f}/{r['ece_10bin']:.3f}"
                     for _, r in run_metrics_df.iterrows()))
    print(f"  {run_name} done in {elapsed:.1f}s")
    print()

    run_metrics_df["run"]  = run_idx + 1
    run_metrics_df["seed"] = run_seed
    master_rows.append(run_metrics_df)

total_elapsed = time.time() - marathon_t0
print("=" * 60)
print(f"30-run hospital-out reproducibility loop complete: {total_elapsed:.0f}s")
print("=" * 60)

30-Run Hospital-Out Reproducibility (BCE loss, GP zero-shot)
  Each run: fresh random 52/13 hospital partition -> retrain LR/RF/XGB/LR_platt/RF_platt
            -> apply canonical GP formula zero-shot (sigmoid, not clip) -> evaluate
  Resume-safe: completed runs are skipped automatically

run_01 (seed=0): starting ...
  Train: 9,277 pts / 52 hosp (17.47%)   Test: 1,887 pts / 13 hosp (13.99%)
  Test metrics (AUROC/ECE): GP=0.746/0.030  LR=0.769/0.270  RF=0.791/0.143  XGB=0.772/0.167  LR_platt=0.769/0.032  RF_platt=0.791/0.033  APACHE-IV=0.692/0.083
  run_01 done in 9.8s

run_02 (seed=1): starting ...
  Train: 8,591 pts / 52 hosp (17.05%)   Test: 2,573 pts / 13 hosp (16.32%)
  Test metrics (AUROC/ECE): GP=0.751/0.012  LR=0.776/0.227  RF=0.777/0.099  XGB=0.753/0.113  LR_platt=0.776/0.018  RF_platt=0.777/0.015  APACHE-IV=0.704/0.051
  run_02 done in 5.3s

run_03 (seed=2): starting ...
  Train: 8,535 pts / 52 hosp (16.88%)   Test: 2,629 pts / 13 hosp (16.89%)
  Test metrics (AUROC/ECE): GP

In [3]:
# ════════════════════════════════════════════════════════════════════════════
# Aggregate all 30 runs into the master reproducibility table.
# Self-contained: reads from saved run_metrics.csv files on disk.
# ════════════════════════════════════════════════════════════════════════════
rows = []
for run_dir in sorted(REPRO_DIR.glob("run_*")):
    mpath = run_dir / "run_metrics.csv"
    if not mpath.exists():
        continue
    df = pd.read_csv(mpath)
    df["run"]  = int(run_dir.name.split("_")[1])
    df["seed"] = int(run_dir.name.split("_")[1]) - 1
    rows.append(df)

if rows:
    master_df = pd.concat(rows, ignore_index=True)
else:
    master_df = pd.concat(master_rows, ignore_index=True)

out_master = TABLES / "NB12_30run_reproducibility.csv"
master_df.to_csv(out_master, index=False)
print(f"Saved: {out_master}")
print(f"  {len(master_df)} rows = {master_df['run'].nunique()} runs x {master_df['model'].nunique()} models")
print()

print("Mean +/- SD across 30 runs, per model:")
summary = (master_df.groupby("model")[
    ["auroc", "auprc", "brier", "ece_10bin", "cal_slope", "cal_intercept"]
].agg(["mean", "std"]))
print(summary.round(4).to_string())

print()
print("Per-run test-hospital partition size and mortality:")
n_runs = master_df["run"].nunique()
for run_idx in range(n_runs):
    run_dir = REPRO_DIR / f"run_{run_idx + 1:02d}"
    if not (run_dir / "hospital_split.csv").exists():
        continue
    sp = pd.read_csv(run_dir / "hospital_split.csv")
    te = sp[sp["split"] == "test"]
    tr = sp[sp["split"] == "train"]
    te_full = feat[feat["patientunitstayid"].isin(te["patientunitstayid"])]
    tr_full = feat[feat["patientunitstayid"].isin(tr["patientunitstayid"])]
    print(f"  run_{run_idx+1:02d}: train n={len(tr_full):,} ({tr_full['hospital_mortality'].mean()*100:.2f}%)  "
          f"test n={len(te_full):,} ({te_full['hospital_mortality'].mean()*100:.2f}%)")

Saved: C:\ML PROJECT\sepsis-gp\results\v2_bce\tables\NB12_30run_reproducibility.csv
  210 rows = 30 runs x 7 models

Mean +/- SD across 30 runs, per model:
            auroc           auprc           brier         ece_10bin         cal_slope         cal_intercept        
             mean     std    mean     std    mean     std      mean     std      mean     std          mean     std
model                                                                                                              
APACHE-IV  0.7017  0.0159  0.3780  0.0332  0.1349  0.0082    0.0612  0.0099    0.4113  0.0773       -0.9501  0.1246
GP         0.7416  0.0139  0.4302  0.0302  0.1216  0.0086    0.0163  0.0059    1.0469  0.0694        0.0818  0.1411
LR         0.7672  0.0101  0.4600  0.0289  0.1912  0.0094    0.2516  0.0138    0.8938  0.0551       -1.5761  0.0995
LR_platt   0.7672  0.0101  0.4600  0.0289  0.1180  0.0078    0.0177  0.0064    0.9970  0.0626       -0.0010  0.1631
RF         0.7768  0.0111  0.465

## Findings
**This notebook is supplementary to `NB12_hospital_out_evaluation.ipynb`.** The primary hospital-out result for this thesis is the canonical 52/13 split in NB12. The findings below assess whether that result is specific to one particular choice of which 13 hospitals are held out, or whether it replicates across different hospital partitions. The canonical GP formula (c=24, seed=14) is held fixed in every run — only the hospital partition and the freshly retrained baselines vary.

### 30-run reproducibility — fresh hospital partitions each time

All 30 partitions completed in 97 seconds (no PySR calls — GP applied zero-shot). Results aggregated to `results/v2_bce/tables/NB12_30run_reproducibility.csv` (210 rows = 30 partitions × 7 models).

| Model | AUROC mean ± SD | ECE mean ± SD | Brier mean ± SD | Cal slope mean ± SD |
|---|---|---|---|---|
| GP | 0.742 ± 0.014 | **0.016 ± 0.006** | 0.122 ± 0.009 | 1.047 ± 0.069 |
| LR | 0.767 ± 0.010 | 0.252 ± 0.014 | 0.191 ± 0.009 | 0.894 ± 0.055 |
| RF | 0.777 ± 0.011 | 0.112 ± 0.014 | 0.131 ± 0.006 | 1.384 ± 0.095 |
| XGB | 0.757 ± 0.010 | 0.134 ± 0.015 | 0.148 ± 0.006 | 0.755 ± 0.041 |
| LR_platt | 0.767 ± 0.010 | **0.018 ± 0.006** | 0.118 ± 0.008 | 0.997 ± 0.063 |
| RF_platt | 0.777 ± 0.011 | **0.019 ± 0.006** | 0.117 ± 0.008 | 0.999 ± 0.071 |
| APACHE-IV | 0.702 ± 0.016 | 0.061 ± 0.010 | 0.135 ± 0.008 | 0.411 ± 0.077 |

**Calibration — the central finding, confirmed at 30-partition scale.** GP (ECE 0.016), LR_platt (0.018), and RF_platt (0.019) form a tight three-way cluster with essentially identical ECE means and standard deviations. All three calibration slopes cluster near unity (1.047, 0.997, 0.999), confirming well-centred probability estimates as a stable property across different hospital compositions. GP achieves this without any post-hoc recalibration step. This cluster is clearly separated from every uncalibrated baseline: RF (0.112), XGB (0.134), LR (0.252), and from APACHE-IV (0.061). The pattern holds without exception across all 30 partitions — no single partition produced a GP calibration outlier.

**Discrimination — GP deficit consistent across partitions.** GP's mean AUROC (0.742) trails RF/RF_platt (0.777, gap 0.035), LR/LR_platt (0.767, gap 0.025), and XGB (0.757, gap 0.015), while clearly outperforming APACHE-IV (0.702, advantage 0.040). The spread across partitions is tight (GP SD 0.014, comparable to other models' 0.010–0.016), confirming the discrimination gap is not driven by any particular partition but is a stable property of the formula across all hospital compositions tested.

### Partition variability — expected, not an error

Hospital partitions were not stratified by size or mortality, so test-set composition varies naturally depending on which 13 hospitals were sampled. Test patient counts ranged from 1,691 (run_12) to 3,071 (run_19), and test-set mortality from 13.81% (run_09) to 19.75% (run_08) — a 5.94 pp spread. Despite this substantial variability in held-out population characteristics, GP's AUROC and ECE remained tightly clustered (SD 0.014 and 0.006 respectively), confirming the canonical RQ2 conclusions are not sensitive to which specific hospitals are held out.

### Summary

The canonical single-partition hospital-out result (NB12 Cell 4) is substantially confirmed across 30 independent hospital partitions: GP's calibration is statistically indistinguishable from LR_platt and RF_platt (mean ECE 0.016 vs 0.018 and 0.019), and clearly better than every uncalibrated baseline and APACHE-IV, as a stable property across a wide range of held-out hospital compositions. This strengthens the answer to RQ2: GP's zero-shot calibration stability under hospital-out validation is reproducible, not an artefact of one particular partition choice.
